<a href="https://colab.research.google.com/github/abhisakh/XGBOOST_Machine_Leraning_Predictor_Titanic/blob/main/Synthetic_Data_Analyzer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# XGBoost Machine Learning Pipeline:
Complete Research Framework
Clinical-Epidemiological Disease Burden & Mortality Prediction
Project Scope
This notebook implements a comprehensive two-path ML pipeline with:

**Path A:** Disease Burden Regression (total_diagnoses count)

**Path B:** Mortality Risk Classification (binary is_deceased)
Advanced Comparisons: 4 different architectural approaches
Performance Leaderboard: ROC-AUC vs PR-AUC rankings

**Interpretability:** SHAP values and feature attribution

# Necessary module import

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error, r2_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report, precision_recall_curve,
    precision_score, recall_score, roc_curve
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
print('✓ All libraries imported')

✓ All libraries imported


# Data Import

In [3]:
url = 'https://raw.githubusercontent.com/abhisakh/XGBOOST_Machine_Leraning_Predictor_Titanic/refs/heads/main/data/NODE_A_unified.csv'
df = pd.read_csv(url)

# Data Overview

In [4]:
df.head()

,id,age,birth_date,sex,region,urban_rural,household_id,household_size,income_quartile,estimated_income,...,total_billing_scheine,total_hospital_stays,total_hospital_cost,total_interventions,avg_treatment_effect,has_counterfactual_record,dmp_program_count,total_dmp_visits,total_switches,is_deceased
0,1,77,1947-07-02,M,NRW,urban,7513,2,Q1,10000.0,...,19.0,0.0,0.00,4.0,0.0,7,0.0,0.0,0.0,0.0
1,2,15,2009-12-13,F,NRW,urban,2417,1,Q2,0.0,...,0.0,0.0,0.00,0.0,0.0,7,0.0,0.0,1.0,0.0
2,3,53,1971-11-10,F,NRW,urban,5723,2,Q4,67643.0,...,1.0,2.0,13818.96,5.0,0.0,7,2.0,26.0,0.0,0.0
3,4,60,1964-07-13,M,NRW,urban,6307,5,Q4,65053.0,...,0.0,0.0,0.00,2.0,0.0,7,0.0,0.0,0.0,0.0
4,5,31,1993-08-07,F,RP,rural,3880,1,Q3,44246.0,...,0.0,0.0,0.00,0.0,0.0,7,0.0,0.0,0.0,0.0


# Column names

In [10]:
print(df.columns)

Index(['id', 'age', 'birth_date', 'sex', 'region', 'urban_rural',
       'household_id', 'household_size', 'income_quartile', 'estimated_income',
       'education', 'employment_status', 'migration_background', 'zip_code',
       'gisd_quintile', 'insurance_fund', 'insurance_status',
       'insurance_start', 'insurance_end', 'exposome_air_quality',
       'exposome_green_space', 'exposome_noise_level', 'exposome_deprivation',
       'total_diagnoses', 'confirmed_diagnoses', 'total_ambulatory_visits',
       'total_prescriptions', 'total_drug_cost', 'total_billing_scheine',
       'total_hospital_stays', 'total_hospital_cost', 'total_interventions',
       'avg_treatment_effect', 'has_counterfactual_record',
       'dmp_program_count', 'total_dmp_visits', 'total_switches',
       'is_deceased'],
      dtype='object')


# 📊 Exploratory Data Analysis (EDA):
<h1>Non-Numeric Features</h1>
<h2>💡 Motivation & Objective</h2>
<p>Machine learning models are mathematical frameworks that require numerical inputs. Raw text labels cannot be parsed directly by our algorithms. Therefore, before we can proceed to model training, we must identify, inspect, and prepare our non-numeric columns.</p>
<br><h3>We perform an inspection of unique categories to achieve three primary goals:</h3>

1. Determine Encoding Strategy: Identify which variablesare binary (two categories), ordinal (ordered categories, like education level), or nominal (unordered text, like region) so we can map them to numbers using the right method (e.g., One-Hot Encoding vs. Ordinal Mapping).
2. Assess Cardinality: Identify columns with high cardinality (too many unique categories). High cardinality can lead to the "curse of dimensionality" if we split them into too many columns, which dramatically slows down training and risks overfitting.
3. Data Quality Check: Uncover data-entry typos (e.g., inconsistent formatting, casing variations) and hidden missing-value placeholders (e.g., strings like "Unknown", "?", or "None") that standard automated scripts might miss.

<h2>📈 Analytical Steps to ExecuteIdentify Data Types:</h2>
<p>Compile a comprehensive list of columns stored as object or text formats.

1. Identify Data Types: Compile a comprehensive list of columns stored as object or text formats.
2. Count and List Unique Labels: Examine the distinct text expressions within each column using .unique() and calculate total counts using .nunique().
3. Feature Engineering Isolation: Flag date-related fields (birth_date, insurance_start, insurance_end) to transform them later into operational numerical durations (such as calculating age or length of insurance coverage).</p>

# 1. Identify Data Types: Columns containing non-numeric values

In [16]:
non_numeric_features = [feature for feature in df.columns if df[feature].dtype == object ]
for feature in non_numeric_features:
  print(feature,'::', df[feature].dtype)

birth_date :: object
sex :: object
region :: object
urban_rural :: object
income_quartile :: object
education :: object
employment_status :: object
migration_background :: object
insurance_fund :: object
insurance_status :: object
insurance_start :: object
insurance_end :: object


# 2. Count and List Unique Labels

In [23]:
non_numeric_features = [feature for feature in df.columns if df[feature].dtype == object]

for feature in non_numeric_features:
    unique_vals = list(df[feature].unique())
    num_unique = df[feature].nunique()

    # If there are more than 5 unique items, show a preview instead of flooding the screen
    if len(unique_vals) > 5:
        preview = f"{unique_vals[:5]} ... [and {len(unique_vals)-5} more]"
    else:
        preview = f"{unique_vals}"

    print(f"{feature:<22} :: {num_unique:<5} categories => {preview}")


birth_date             :: 11619 categories => ['1947-07-02', '2009-12-13', '1971-11-10', '1964-07-13', '1993-08-07'] ... [and 11614 more]
sex                    :: 2     categories => ['M', 'F']
region                 :: 16    categories => ['NRW', 'RP', 'BW', 'BE', 'NI'] ... [and 11 more]
urban_rural            :: 2     categories => ['urban', 'rural']
income_quartile        :: 4     categories => ['Q1', 'Q2', 'Q4', 'Q3']
education              :: 8     categories => ['ISCED_5', 'ISCED_2', 'ISCED_6', 'ISCED_3', 'ISCED_4'] ... [and 3 more]
employment_status      :: 4     categories => ['retired', 'other', 'employed', 'unemployed']
migration_background   :: 3     categories => ['second_generation', 'none', 'first_generation']
insurance_fund         :: 1     categories => ['NODE_A']
insurance_status       :: 4     categories => ['KVdR', 'Familienversichert', 'Pflichtmitglied', 'Freiwillig Versicherter']
insurance_start        :: 2188  categories => ['2019-07-31', '2020-08-08', '2023-08-0

<h1>🗑️ 1. Drop Invariant Columns (Zero Information)</h1>
<p>
insurance_fund (1 category):
Since every single row is 'NODE_A', this column provides zero variance or predictive power.Action: Drop it completely using df.drop(columns=['insurance_fund'], inplace=True).</p>
<h1>📅 2. Transform Date Fields into Durationsbirth_date (11,619 categories):</h1>
<p>Do not encode this as a category. Calculate the patient's age relative to a target reference date or current year.insurance_start & insurance_end: Calculate the actual coverage duration by subtracting start from end: insurance_end - insurance_start.</p>

<h1>🏷️ 3. Encode Categories with 2 Options (Binary Mapping)sex & urban_rural (2 categories):</h1>
<p>
Perfect for simple binary mapping to 0 and 1. <br> Action: Map 'M'/'urban' to 1 and 'F'/'rural' to 0.</p>

<h1>📶 4. Ordinal Categories (Ordered Levels)income_quartile (4 categories):</h1>
<p>
 These have a natural mathematical order (Q1 < Q2 < Q3 < Q4).<br>Action: Map them explicitly to integers: {'Q1': 1, 'Q2': 2, 'Q3': 3, 'Q4': 4}.education (8 categories):<br> These follow the international ISCED standard. Higher numbers mean a higher level of education (e.g., ISCED 6 is a Bachelor's level, ISCED 2 is lower secondary). We should map these ordinally based on their numeric value.</p>

 <h1>🗺️ 5. Nominal Categories (Unordered Levels)region (16 categories):</h1>
 <p>These are the 16 German federal states (e.g., NRW = North Rhine-Westphalia, BW = Baden-Württemberg).employment_status, migration_background, insurance_status: Unordered textual buckets.<br>Action: Use One-Hot Encoding (pd.get_dummies()) for these columns since their unique value counts are low enough to avoid hurting your model's speed.</p>